<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_Simulation_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

## Middle East Oil Security Simulation Engine
MOS is an LLM-driven geopolitical simulation in which strategic actors choose actions within an explicitly represented and evolving WorldState. Chanakya proposes actor-specific strategic options, Centaur evaluates their relative likelihood, and ZeitWorld translates the selected Event into the next WorldState. The objective is not to predict the future, but to explore plausible, path-dependent trajectories arising from alternative strategic choices.

In [ ]:
!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mosZeitWorld.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosZeitWorld.py
!wget -O mosChanakya.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosChanakya.py

from mos import *
from mosZeitWorld import *
from mosChanakya import *

In [ ]:
import importlib

import mos
import mosZeitWorld
import mosChanakya

importlib.reload(mos)
importlib.reload(mosZeitWorld)
importlib.reload(mosChanakya)

from mos import *
from ZeitWorld import *
from mosChanakya import *

In [ ]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com


## ZeitWorld - when Events change Worlds

In [ ]:
# These are the eight variables that currently define the World
# This number can be increased if necessasary
#
# ---------------------------------------------------------------

worldVariables = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction="Higher index means higher oil price."
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction="Higher index means greater oil flow."
    ),

    WorldVariable(
        name="India Gulf Oil Imports",
        description=(
            "Volume of crude oil imported by India from "
            "Gulf-origin oil-producing countries."
        ),
        direction=(
            "Higher index means greater Indian imports "
            "of Gulf-origin oil."
        )
    ),

    WorldVariable(
        name="Iran Sanctions Enforcement",
        description=(
            "Degree to which restrictions on Iranian petroleum exports, "
            "trade and financial transactions are effectively enforced "
            "rather than circumvented."
        ),
        direction=(
            "Higher index means more effective sanctions enforcement "
            "and less successful circumvention."
        )
    ),

    WorldVariable(
        name="US/Allied Cohesion",
        description=(
            "Degree of alignment between the United States and cooperating "
            "states in their response to Iran and the Gulf crisis."
        ),
        direction="Higher index means greater US/allied cohesion."
    ),

    WorldVariable(
        name="Iran Internal Cohesion",
        description=(
            "Degree of alignment within Iran's political, military and "
            "security establishment in responding to the crisis."
        ),
        direction="Higher index means greater internal Iranian cohesion."
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction="Higher index means greater US-Iran tension."
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )
]

In [ ]:
# Historical Data and
# LLM Roles and prompts are stored and retrieved from Google Drive
#
# ----------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
WORLDFACTS_DIR = f"{MOS_HOME}/WorldFacts"
ZW_PATH = f"{MOS_HOME}/ZeitWorld"
CH_PATH = f"{MOS_HOME}/Chanakya"

!ls -l "{WORLDFACTS_DIR}"
!ls -lh "{WORLDFACTS_DIR}/MD"
!ls -lh "{WORLDFACTS_DIR}/NEWS"
!ls -lh "{ZW_PATH}"
!ls -lh "{CH_PATH}"

Mounted at /content/drive
total 12
drwx------ 2 root root 4096 Sep 28 09:09 MD
drwx------ 2 root root 4096 Sep 29 09:42 NEWS
drwx------ 2 root root 4096 Sep 28 06:55 PDF
total 163K
-rw------- 1 root root 7.5K Oct  2 07:03 IranData_261002.md
-rw------- 1 root root 5.6K Oct  2 07:03 MissilesofIranMissil_261002.md
-rw------- 1 root root 8.3K Oct  2 07:04 OilMarketReportSepte_261002.md
-rw------- 1 root root  12K Oct  2 07:04 PressReleasePagePres_261002.md
-rw------- 1 root root 4.3K Oct  2 07:04 ShortTermEnergyOutlo_261002.md
-rw------- 1 root root  37K Oct  2 07:06 SnapshotofIndiasOilG_261002.md
-rw------- 1 root root  18K Oct  2 07:06 UKCommonsIranin2026S_261002.md
-rw------- 1 root root  70K Oct  2 07:08 USCongressR488873_261002.md
total 71K
-rw------- 1 root root 71K Oct  2 07:14 MOS-MC-LiveFeed_261002_1244_IST.txt
total 13K
-rw------- 1 root root 2.2K Oct  1 00:10 ZW_Initialisation_Prompt_v0.txt
-rw------- 1 root root 3.1K Oct  1 14:08 ZW_Initialisation_Prompt_v1.txt
-rw------- 1 roo

In [ ]:
# Knowledge of the initial world state is collected from a
# a set of PDF files that have been converted to markdown format
# live news data from 1 source, viz. MoneyControl.com
# Assembled into one large corpus
#
# ------------------------------------------------

WorldFactBase = getWorldFactBase(WORLDFACTS_DIR)


Static sources : 8
Live sources   : 1
Characters     : 235,072
Words          : 35,600


In [ ]:
# Zeitworld Roles and Prompts are retrieved from Google Drive

ZW_Role = getPrompt(f"{ZW_PATH}/ZW_Role_v0.txt")

#ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v0.txt")
ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v1.txt")

ZW_Update_Prompt = getPrompt(f"{ZW_PATH}/ZW_Update_Prompt_v0.txt")

In [ ]:
# ZeitWorld object is instantiated from its class along with its ROLE
#
ZeitWorld = ZeitWorld_Agent(client, model, ZW_Role)

#----------
print(ZeitWorld.initialised)
#print(ZeitWorld.model)

False


In [ ]:
# W is a list of evolving World States
# An empty list is created. A blank world template is created, without any information
# ----------------------------

W = []

W0_Template = WorldState(
    state_id="W0",
    variables=worldVariables
)

In [ ]:
# ZeitWorld reads WorldFactBase data corpus and creates a World State based on World Template
# This initial World State is inserted into the World List W and becomes W[0]
#
# ----------------------------------------

_ = ZeitWorld.initialise(W,W0_Template, WorldFactBase, ZW_Initialisation_Prompt)

print(ZeitWorld.initialised)

Initialising WorldState ... 
Model: gpt-5.6-luna | Effort: medium | Input: 57,064 | Output: 1,856 | Reasoning: 88 | Total: 58,920
True


In [ ]:
# A first look at the intial World State W[0]
# The initial values are all set to 100, as the baseline
#

#W[0].show(-2)  # ---- just the world vectors
#W[0].show()    # ---- the entire state
#W[0].show(4)    # ---- details of any one vector

## Epoch, Events and Chanakya

In [ ]:
# ============================================================
# Load Chanakya Roles
# ============================================================

#-rw------- 1 root root 2.0K Oct  2 00:06 CH_Role_Generic_v0.txt
#-rw------- 1 root root 1.8K Oct  2 00:07 CH_Role_Iran_v0.txt
#-rw------- 1 root root 1.8K Oct  2 00:07 CH_Role_US_v0.txt

CH_Generic_Role = getPrompt(f"{CH_PATH}/CH_Role_Generic_v0.txt")
CH_US_Role      = getPrompt(f"{CH_PATH}/CH_Role_US_v0.txt")
CH_IR_Role      = getPrompt(f"{CH_PATH}/CH_Role_Iran_v0.txt")
CH_BinaryHD     = getPrompt(f"{CH_PATH}/CH_Strategy_BinaryHD_v0.txt")

In [ ]:
# Instantiate Chanakya objects with their own
# National Actor/National interests
# Strategy -- currently only Binary Hawk / Dove

Chanakya_US = Chanakya_Agent(
    agent_id="ChanakyaUS_HD",
    actor="US",
    generic_role=CH_Generic_Role,
    specific_role=CH_US_Role,
    strategy=CH_BinaryHD
)

Chanakya_US.show()

Chanakya_IR = Chanakya_Agent(
    agent_id="ChanakyaIR_HD",
    actor="Iran",
    generic_role=CH_Generic_Role,
    specific_role=CH_IR_Role,
    strategy=CH_BinaryHD
)

Chanakya_IR.show()

ChanakyaRegistry = {
    "US": Chanakya_US,
    "Iran": Chanakya_IR
}

Chanakya : ChanakyaUS_HD
Actor    : US
Chanakya : ChanakyaIR_HD
Actor    : Iran


In [ ]:
# Two optional Triggers, we manually choose any ONE, to start the process

Trigger1 = Event(
    actor="Iran",
    action=(
        "Iran declares the Strait of Hormuz closed to all commercial shipping, "
        "orders its forces to prevent tanker transit, and launches attacks against "
        "US naval forces operating in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


Trigger2 = Event(
    actor="US",
    action=(
        "The United States announces an immediate suspension of offensive military "
        "operations against Iran and offers direct negotiations on maritime security "
        "and sanctions relief, while maintaining defensive protection of US forces "
        "and commercial shipping in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


In [ ]:
# Choose the starting trigger event for initial epoch
DefaultTrigger = Trigger1
#Trigger = Trigger2

In [ ]:
# E is a list of events that will define History
#
E = []

### Epoch - this will loop

In [ ]:
# ============================================================
# Establish Current Trigger
# ============================================================

epoch = len(W) - 1
if len(E) == 0:
    CurrentTrigger = DefaultTrigger
    CurrentTrigger.event_id = f"E{epoch}"
    E.append(CurrentTrigger)
else:
    CurrentTrigger = E[-1]

E[epoch].show()

Event     : E0
Actor     : Iran
Action    : Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.
Rationale : Initial Trigger for Iran


In [ ]:
print("epoch :", epoch)
print("len(W):", len(W))
print("len(E):", len(E))

epoch : 0
len(W): 1
len(E): 1


In [ ]:
# ZeitWorld updates the world [W], moving it from W[epoch] to W[epoch+1] using Trigger Event E[epoch]
#
# -------------------------------

_ = ZeitWorld.update( W,E[epoch],ZW_Update_Prompt)

E[epoch].show()
W[epoch+1].show()

Updating WorldState ---------- W0
Model: gpt-5.6-luna | Effort: medium | Input: 3,083 | Output: 2,404 | Reasoning: 246 | Total: 5,487
Appended  WorldState ---------- W1
Event     : E0
Actor     : Iran
Action    : Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.
Rationale : Initial Trigger for Iran

World State : W1
-------------------------------------------------------
 0  Oil Price                          115.00
 1  Hormuz Oil Flow                     60.00
 2  India Gulf Oil Imports              94.00
 3  Iran Sanctions Enforcement         100.00
 4  US/Allied Cohesion                 104.00
 5  Iran Internal Cohesion             100.00
 6  US-Iran Tension                    120.00
 7  Regional Armed Conflict            116.00


In [ ]:
# ============================================================
# Turn Controller
# Who will act next ? Which Actor/Nation ?
# ============================================================

#ActiveChanakya = Chanakya_US
#ActiveChanakya = getActiveChanakya()
ActiveChanakya = getActiveChanakya(E[-1], ChanakyaRegistry)

ActiveChanakya.show()

Chanakya : ChanakyaUS_HD
Actor    : US


In [ ]:
print(epoch)
E[epoch].show()
W[epoch+1].show()

0
Event     : E0
Actor     : Iran
Action    : Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.
Rationale : Initial Trigger for Iran

World State : W1
-------------------------------------------------------
 0  Oil Price                          115.00
 1  Hormuz Oil Flow                     60.00
 2  India Gulf Oil Imports              94.00
 3  Iran Sanctions Enforcement         100.00
 4  US/Allied Cohesion                 104.00
 5  Iran Internal Cohesion             100.00
 6  US-Iran Tension                    120.00
 7  Regional Armed Conflict            116.00


In [ ]:
print("epoch :", epoch)
print("len(W):", len(W))
print("len(E):", len(E))

epoch : 0
len(W): 2
len(E): 1


In [ ]:
# ============================================================
# Generate Candidate Reaction Events
# ============================================================

Candidates = ActiveChanakya.propose(
    client=client,
    model=model,
    trigger=E[epoch],
    world=W[epoch+1]
)

Chanakya working ---------- ChanakyaUS_HD
Model: gpt-5.6-luna | Effort: medium | Input: 2,352 | Output: 334 | Reasoning: 85 | Total: 2,686


In [ ]:
for i, candidate in enumerate(Candidates):
    print(f"\nCandidate Reaction {i+1}")
    print("-" * 60)
    candidate.show()


Candidate Reaction 1
------------------------------------------------------------
Event     : 
Actor     : US
Action    : Expand defensive and coercive maritime operations by striking Iranian forces directly involved in attacks on U.S. naval units and the attempted closure, while reinforcing the blockade, escorting commercial shipping, and issuing an ultimatum requiring Iran to cease interference with tanker transit.
Rationale : A forceful response could restore deterrence after direct attacks on U.S. forces, protect freedom of navigation, and impose immediate costs on Iran's effort to use Hormuz as leverage. Coordinated action with Gulf partners could also demonstrate alliance resolve and reassure threatened shipping interests.

Candidate Reaction 2
------------------------------------------------------------
Event     : 
Actor     : US
Action    : Adopt a temporary risk-reduction posture centered on defensive protection of U.S. and commercial shipping, while using intermediaries and

In [ ]:
#print(Chanakya_US._event_to_text(E[t]))
#print()
#print(Chanakya_US._world_to_text(W[t+1]))

In [ ]:
Reaction = Candidates[0]     # manual selection for now
Reaction.event_id = f"E{len(E)}"
E.append(Reaction)

In [ ]:
print(Reaction.event_id)
print(Reaction.actor)
print(Reaction.action)
print(Reaction.rationale)

E1
US
Expand defensive and coercive maritime operations by striking Iranian forces directly involved in attacks on U.S. naval units and the attempted closure, while reinforcing the blockade, escorting commercial shipping, and issuing an ultimatum requiring Iran to cease interference with tanker transit.
A forceful response could restore deterrence after direct attacks on U.S. forces, protect freedom of navigation, and impose immediate costs on Iran's effort to use Hormuz as leverage. Coordinated action with Gulf partners could also demonstrate alliance resolve and reassure threatened shipping interests.


In [ ]:
print("epoch :", epoch)
print("len(W):", len(W))
print("len(E):", len(E))

epoch : 0
len(W): 2
len(E): 2


### Epoch Loop Ends

In [ ]:
def resetWorld():
    global ActiveChanakya, Candidates
    del W[1:]
    E.clear()
    ActiveChanakya = None
    Candidates = []

    print("World reset to W0.")
# ------------------------
resetWorld()

World reset to W0.


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)